# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [7]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [8]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'portfolio / projects',
   'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'portfolio / projects',
   'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'portfolio / projects',
   'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'portfolio / projects',
   'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'blog', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'LinkedIn', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook', 'url': 'https://www.facebook.com/edward.donner.52'}]}

In [ ]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [9]:
select_relevant_links("https://huggingface.co")

{'links': [{'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'community forum', 'url': 'https://discuss.huggingface.co'},
  {'type': 'GitHub', 'url': 'https://github.com/huggingface'},
  {'type': 'Twitter', 'url': 'https://twitter.com/huggingface'},
  {'type': 'LinkedIn', 'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'product endpoints', 'url': 'https://endpoints.huggingface.co'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [10]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [11]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
Qwen/Qwen3.6-35B-A3B
Updated
about 19 hours ago
•
583k
•
1.26k
moonshotai/Kimi-K2.6
Updated
2 days ago
•
54.5k
•
819
unsloth/Qwen3.6-35B-A3B-GGUF
Updated
3 days ago
•
1.11M
•
675
Qwen/Qwen3.6-27B
Updated
about 16 hours ago
•
471
tencent/HY-World-2.0
Updated
about 20 hours ago
•
555
Browse 2M+ models
Spaces
Running
on
Zero
MCP
2.31k
Wan2.2 14B Preview
🐌
2.31k
generate a video from an image with a text prompt
Running
on
Zero
Agents
Featured
661
OmniVoice
🌍
661
High-quality voice cloning TTS for 600+ languages
Running
156
Bonsai 1-bit WebGPU
🌳
156
Run 1-bit Bonsai LLMs locally in your browser on WebGPU
Running
on
Zero
MCP
Featured

In [12]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [13]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [14]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nQwen/Qwen3.6-35B-A3B\nUpdated\nabout 19 hours ago\n•\n583k\n•\n1.26k\nmoonshotai/Kimi-K2.6\nUpdated\n2 days ago\n•\n54.5k\n•\n820\nunsloth/Qwen3.6-35B-A3B-GGUF\nUpdated\n3 days ago\n•\n1.11M\n•\n676\nQwen/Qwen3.6-27B\nUpdated\nabout 16 hours ago\n•\n474\ntencent/HY-World-2.0\nUpdated\nabout 20 hours ago\n•\n555\nBrowse 2M+ models\nSpaces\nRunning\non\nZero\nMCP\n2.31k\nWan2.2 14B Preview

In [15]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [16]:
create_brochure("HuggingFace", "https://huggingface.co")

# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is the leading collaboration platform for the machine learning (ML) community, dedicated to building the future of AI together. It serves as a central hub where ML engineers, scientists, and end users can share, explore, discover, and experiment with open-source machine learning models, datasets, and applications. Hugging Face empowers the next generation in AI by fostering an open and ethical AI ecosystem driven by collaboration.

With a rapidly growing global community, Hugging Face hosts over **2 million models**, **500,000+ datasets**, and **1 million+ AI applications**, making it one of the largest open-source AI resources worldwide. Their platform embraces all modalities of machine learning—including text, image, video, audio, and 3D—helping users to create, build portfolios, and accelerate their AI projects.

---

## Core Offerings

- **Models & Datasets:** Access and contribute to millions of pre-trained ML models and open datasets.
- **Spaces:** Run and share ML applications and demos in an interactive environment.
- **Compute & Enterprise Solutions:** Scalable compute resources and tailored enterprise plans for teams of all sizes.
- **Open Source Tools:** A comprehensive, well-maintained open-source stack that speeds up ML development and deployment.

---

## Enterprise & Team Plans

Hugging Face offers flexible plans designed for organizations looking to scale their AI capabilities with

- **Enterprise-grade security & access controls:** Single Sign-On (SSO), audit logs, and granular permission management.
- **Advanced compute options:** Including ZeroGPU with quota boosts for enhanced performance.
- **Private storage & dataset viewers:** Secure your proprietary data and collaborate easily.
- **Analytics dashboard:** Gain insights on repository usage and team productivity.
- **Dedicated support & billing:** Customized contracts and integration support.

Pricing starts at $20/user/month for teams, with enterprise contracts available for large organizations.

---

## Company Culture & Community

Hugging Face thrives on community-driven collaboration and transparency, emphasizing open source and ethical AI practices. They actively foster a culture that:

- Encourages knowledge sharing, continuous learning, and innovation.
- Welcomes contributors from diverse backgrounds to co-create ML models and datasets.
- Supports growth of individual ML portfolios by enabling public sharing of work.
- Drives the frontier of AI research through a talented, passionate science team.

---

## Careers & Opportunities

Hugging Face is constantly growing and looking for talented individuals passionate about AI and open source to join their team. Current openings span across:

- Research and Engineering
- Product and Design
- Community and Developer Advocacy
- Enterprise Sales and Support

Working at Hugging Face means contributing to cutting-edge AI technologies in a collaborative, mission-driven environment.

---

## Join the AI Revolution

Whether you are an ML pioneer, an enterprise scaling AI capabilities, or a developer eager to learn, Hugging Face offers the tools, community, and infrastructure to build the future of artificial intelligence together.

**Discover more and get started at [huggingface.co](https://huggingface.co).**

---

**Contact & Socials:**  
GitHub · Twitter · LinkedIn · Discord

---

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [18]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [ ]:
stream_brochure("HuggingFace", "https://huggingface.co")

In [19]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("Himanshu Puri", "https://www.linkedin.com/in/himanshupuri/")

# Himanshu Puri | MSCI Inc. - Company Brochure

## About Himanshu Puri and MSCI Inc.

Himanshu Puri is an accomplished IT professional with 26 years of experience in the financial services industry, currently associated with MSCI Inc., a global leader in critical decision support tools and services tailored for the investment community.

### MSCI Inc. Overview

- **Industry:** Financial Services
- **Headquarters:** New York, NY
- **Company Size:** 1,001 - 5,000 employees
- **Followers on LinkedIn:** 348,570
- **Website:** [www.msci.com](http://www.msci.com)

### Our Mission

MSCI empowers investors worldwide by providing research-enhanced solutions that improve transparency and insight into investment risks and returns. Our clients build more effective portfolios with confidence, thanks to over 50 years of expertise in research, data, and technology.

### What We Do

- Index Solutions
- Risk Analytics
- Portfolio Analytics
- ESG (Environmental, Social, Governance) Research
- Real Estate Analytics
- Factor Investing

MSCI’s suite of products enables the global investment community to understand and analyze key drivers of risk and return, optimizing investment decisions.

---

## Company Culture

At MSCI, diverse perspectives fuel innovation. We foster an inclusive workplace where curiosity, integrity, and excellence thrive. Our culture encourages collaboration across our global teams to deliver industry-leading solutions that drive impact for clients.

---

## Customers

MSCI serves asset managers, banks, pension funds, insurers, hedge funds, and other institutional investors globally. Our clients rely on us for transparent, research-driven data and analytics to navigate complex financial markets and create sustainable investment strategies.

---

## Careers at MSCI

Join the MSCI team if you want to work at the forefront of financial technology, research, and sustainability. With over 8,000 employees worldwide, MSCI offers opportunities in:

- Data Science and Analytics
- Software Engineering
- Research and Portfolio Management
- Client Services and Product Management
- ESG and Real Estate Specialties

We seek passionate professionals eager to contribute to a dynamic, growth-oriented environment.

---

## Connect with Us

- LinkedIn Profile of Himanshu Puri: Active with 1K+ followers, well-connected in MSCI’s professional network.
- MSCI LinkedIn Page: [Follow Us](https://www.linkedin.com/company/msci)
- Visit our website for career opportunities and latest updates: [www.msci.com](http://www.msci.com)

---

**Empowering the global investment community through data, technology, and insights — MSCI Inc.**

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>